In [1]:
pip install duckdb

  Obtaining dependency information for duckdb from https://files.pythonhosted.org/packages/46/59/a8e3384ee916e00d5dcf985194c1511d61978540778a1e96fa47f9fb3e0d/duckdb-1.5.5-cp311-cp311-macosx_11_0_arm64.whl.metadata
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.5/15.5 MB 10.1 MB/s eta 0:00:0000:0100:01

[notice] A new release of pip is available: 23.2.1 -> 26.2.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import duckdb

In [3]:
con = duckdb.connect()

# Структура событий и бронирований

In [4]:
result = con.execute("""
    SELECT COUNT(*) AS rows_count
    FROM 'train.parquet'
""").fetchdf()

result

,rows_count
0,37670293


In [5]:
schema = con.execute("""
    DESCRIBE SELECT *
    FROM 'train.parquet'
""").fetchdf()

schema

,column_name,column_type,null,key,default,extra
0,date_time,VARCHAR,YES,None,None,None
1,site_name,BIGINT,YES,None,None,None
2,posa_continent,BIGINT,YES,None,None,None
3,user_location_country,BIGINT,YES,None,None,None
4,user_location_region,BIGINT,YES,None,None,None
5,user_location_city,BIGINT,YES,None,None,None
6,orig_destination_distance,DOUBLE,YES,None,None,None
7,user_id,BIGINT,YES,None,None,None
8,is_mobile,BIGINT,YES,None,None,None
9,is_package,BIGINT,YES,None,None,None


In [6]:
sample = con.execute("""
    SELECT *
    FROM 'train.parquet'
    LIMIT 5
""").fetchdf()

sample

,date_time,site_name,posa_continent,user_location_country,user_location_region,user_location_city,orig_destination_distance,user_id,is_mobile,is_package,...,srch_children_cnt,srch_rm_cnt,srch_destination_id,srch_destination_type_id,is_booking,cnt,hotel_continent,hotel_country,hotel_market,hotel_cluster
0,2014-08-11 07:46:59,2,3,66,348,48862,2234.2641,12,0,1,...,0,1,8250,1,0,3,2,50,628,1
1,2014-08-11 08:22:12,2,3,66,348,48862,2234.2641,12,0,1,...,0,1,8250,1,1,1,2,50,628,1
2,2014-08-11 08:24:33,2,3,66,348,48862,2234.2641,12,0,0,...,0,1,8250,1,0,1,2,50,628,1
3,2014-08-09 18:05:16,2,3,66,442,35390,913.1932,93,0,0,...,0,1,14984,1,0,1,2,50,1457,80
4,2014-08-09 18:08:18,2,3,66,442,35390,913.6259,93,0,0,...,0,1,14984,1,0,1,2,50,1457,21


In [7]:
con.execute("""
    SELECT
        is_booking,
        COUNT(*) AS cnt
    FROM 'train.parquet'
    GROUP BY is_booking
""").fetchdf()

,is_booking,cnt
0,0,34669600
1,1,3000693


In [8]:
con.execute("""
    SELECT COUNT(DISTINCT user_id) AS users
    FROM 'train.parquet'
""").fetchdf()

,users
0,1198786


In [9]:
event_stats = con.execute("""
    SELECT
        is_booking,
        COUNT(*) AS rows_count,
        SUM(cnt) AS cnt_sum,
        AVG(cnt) AS avg_cnt,
        MIN(cnt) AS min_cnt,
        MAX(cnt) AS max_cnt
    FROM 'train.parquet'
    GROUP BY is_booking
    ORDER BY is_booking
""").fetchdf()

event_stats

,is_booking,rows_count,cnt_sum,avg_cnt,min_cnt,max_cnt
0,0,34669600,52833029.0,1.523901,1,269
1,1,3000693,3046478.0,1.015258,1,21


In [11]:
cnt_stats = con.execute("""
    SELECT
        cnt,
        COUNT(*) AS rows_count,
        ROUND(
            100.0 * COUNT(*) / SUM(COUNT(*)) OVER (),
            3
        ) AS rows_percentage
    FROM 'train.parquet'
    GROUP BY cnt
    ORDER BY cnt
    LIMIT 20
""").fetchdf()

cnt_stats

,cnt,rows_count,rows_percentage
0,1,28091115,74.571
1,2,5686424,15.095
2,3,2024726,5.375
3,4,842356,2.236
4,5,426092,1.131
5,6,224373,0.596
6,7,132234,0.351
7,8,78360,0.208
8,9,51009,0.135
9,10,32763,0.087


In [12]:
booking_rows_stats = con.execute("""
    SELECT
        COUNT(*) AS total_rows,

        SUM(CASE WHEN is_booking = 0 THEN 1 ELSE 0 END) AS click_rows,

        SUM(CASE WHEN is_booking = 1 THEN 1 ELSE 0 END) AS booking_rows,

        ROUND(
            100.0 * SUM(CASE WHEN is_booking = 1 THEN 1 ELSE 0 END)
            / COUNT(*),
            2
        ) AS booking_rows_percentage

    FROM 'train.parquet'
""").fetchdf()

booking_rows_stats

,total_rows,click_rows,booking_rows,booking_rows_percentage
0,37670293,34669600.0,3000693.0,7.97


In [13]:
cnt_stats = con.execute("""
    SELECT
        SUM(cnt) AS total_events,

        SUM(CASE WHEN is_booking = 0 THEN cnt ELSE 0 END) AS click_events,

        SUM(CASE WHEN is_booking = 1 THEN cnt ELSE 0 END) AS booking_events,

        ROUND(
            100.0 * SUM(CASE WHEN is_booking = 1 THEN cnt ELSE 0 END)
            / SUM(cnt),
            2
        ) AS booking_events_percentage,

        SUM(CASE WHEN cnt > 1 THEN 1 ELSE 0 END) AS rows_with_multiple_events,

        ROUND(
            100.0 * SUM(CASE WHEN cnt > 1 THEN 1 ELSE 0 END)
            / COUNT(*),
            2
        ) AS rows_with_multiple_events_percentage

    FROM 'train.parquet'
""").fetchdf()

cnt_stats

,total_events,click_events,booking_events,booking_events_percentage,rows_with_multiple_events,rows_with_multiple_events_percentage
0,55879507.0,52833029.0,3046478.0,5.45,9579178.0,25.43


В датасете содержится 37,7 млн записей о взаимодействиях пользователей с отелями, соответствующих 55,9 млн событий с учётом поля cnt. Около четверти записей (25,43%) агрегируют более одного события. Бронирования составляют 7,97% записей и 5,45% всех событий с учётом cnt. Поэтому при дальнейшем анализе необходимо различать количество записей и фактическое количество событий.

# Качество данных

Сначала проверим, в каких столбцах есть NULL и какую долю датасета они занимают.

In [14]:
columns = con.execute("""
    DESCRIBE SELECT *
    FROM 'train.parquet'
""").fetchdf()["column_name"].tolist()


missing_expressions = ",\n".join(
    [
        f'COUNT(*) - COUNT("{column}") AS "{column}"'
        for column in columns
    ]
)

missing_counts = con.execute(f"""
    SELECT
        COUNT(*) AS total_rows,
        {missing_expressions}
    FROM 'train.parquet'
""").fetchdf()


total_rows = missing_counts["total_rows"].iloc[0]

missing_stats = (
    missing_counts
    .drop(columns="total_rows")
    .T
    .reset_index()
)

missing_stats.columns = [
    "column",
    "missing_count"
]

missing_stats["missing_percentage"] = (
    missing_stats["missing_count"]
    / total_rows
    * 100
).round(2)

missing_stats = missing_stats.sort_values(
    "missing_percentage",
    ascending=False
)

missing_stats

,column,missing_count,missing_percentage
6,orig_destination_distance,13525001,35.90
12,srch_co,47084,0.12
11,srch_ci,47083,0.12
13,srch_adults_cnt,0,0.00
22,hotel_market,0,0.00
21,hotel_country,0,0.00
20,hotel_continent,0,0.00
19,cnt,0,0.00
18,is_booking,0,0.00
17,srch_destination_type_id,0,0.00


Теперь посмотрим диапазоны дат и логические ошибки.

In [15]:
date_stats = con.execute("""
    SELECT
        MIN(date_time) AS min_date_time,
        MAX(date_time) AS max_date_time,

        MIN(srch_ci) AS min_check_in,
        MAX(srch_ci) AS max_check_in,

        MIN(srch_co) AS min_check_out,
        MAX(srch_co) AS max_check_out

    FROM 'train.parquet'
""").fetchdf()

date_stats

,min_date_time,max_date_time,min_check_in,max_check_in,min_check_out,max_check_out
0,2013-01-07 00:00:02,2014-12-31 23:59:59,2012-02-15,2558-03-15,2012-09-04,2558-03-16


In [18]:
check_in_years = con.execute("""
    SELECT
        YEAR(TRY_CAST(srch_ci AS DATE)) AS check_in_year,
        COUNT(*) AS rows_count,
        ROUND(
            100.0 * COUNT(*) / SUM(COUNT(*)) OVER (),
            4
        ) AS rows_percentage
    FROM 'train.parquet'
    WHERE TRY_CAST(srch_ci AS DATE) IS NOT NULL
    GROUP BY YEAR(TRY_CAST(srch_ci AS DATE))
    ORDER BY check_in_year
""").fetchdf()

check_in_years

,check_in_year,rows_count,rows_percentage
0,2012,11,0.0000
1,2013,9853622,26.1903
2,2014,23427909,62.2698
3,2015,4338918,11.5326
4,2016,2703,0.0072
5,2017,4,0.0000
6,2018,9,0.0000
7,2019,2,0.0000
8,2020,2,0.0000
9,2021,1,0.0000


Основная масса дат заезда выглядит правдоподобно:

2013 — 9 853 622 строк, 26.19%
2014 — 23 427 909 строк, 62.27%
2015 — 4 338 918 строк, 11.53%

То есть почти весь датасет укладывается в логичный период 2013–2015. 2016 встречается всего в 2703 строках, а всё после 2016 года — буквально единичные случаи. Даты вроде 2057, 2557, 2558 точно выглядят как ошибки.

Ещё интересны 11 строк с 2012 годом: они тоже потенциально подозрительные, потому что сами события date_time начинаются только с 2013 года. 


In [19]:
check_out_years = con.execute("""
    SELECT
        YEAR(TRY_CAST(srch_co AS DATE)) AS check_out_year,
        COUNT(*) AS rows_count,
        ROUND(
            100.0 * COUNT(*) / SUM(COUNT(*)) OVER (),
            4
        ) AS rows_percentage
    FROM 'train.parquet'
    WHERE TRY_CAST(srch_co AS DATE) IS NOT NULL
    GROUP BY YEAR(TRY_CAST(srch_co AS DATE))
    ORDER BY check_out_year
""").fetchdf()

check_out_years

,check_out_year,rows_count,rows_percentage
0,2012,5,0.0000
1,2013,9620529,25.5707
2,2014,22969011,61.0501
3,2015,5028591,13.3657
4,2016,5029,0.0134
5,2017,2,0.0000
6,2018,2,0.0000
7,2019,9,0.0000
8,2020,2,0.0000
9,2022,1,0.0000


Да, по srch_co картина такая же: основная масса дат выезда сосредоточена в нормальном диапазоне 2013–2015.

То есть проблема есть, но она затрагивает очень малую долю данных.

Теперь важнее проверить не просто годы, а логическую согласованность дат.

сколько случаев, где дата заезда раньше даты действия пользователя:

In [22]:
check_in_before_search = con.execute("""
    SELECT
        COUNT(*) AS invalid_rows,

        ROUND(
            100.0 * COUNT(*) /
            (SELECT COUNT(*) FROM 'train.parquet'),
            4
        ) AS invalid_rows_percentage

    FROM 'train.parquet'
    WHERE TRY_CAST(srch_ci AS DATE) < CAST(date_time AS DATE)
""").fetchdf()

check_in_before_search

,invalid_rows,invalid_rows_percentage
0,8457,0.0225


Обнаружено 8 457 записей, в которых дата заезда предшествует дате пользовательского события. Такие наблюдения логически некорректны и должны исключаться при расчёте времени между поиском и заездом.

При этом сами строки необязательно удалять из датасета целиком: например, для анализа is_mobile или channel они ещё могут быть пригодны. Мы просто не будем использовать их в метриках, зависящих от дат.

Теперь проверим, есть ли случаи, когда выезд раньше или совпадает с датой заезда:

In [23]:
check_out_not_after_check_in = con.execute("""
    SELECT
        COUNT(*) AS invalid_rows,

        ROUND(
            100.0 * COUNT(*) /
            (SELECT COUNT(*) FROM 'train.parquet'),
            4
        ) AS invalid_rows_percentage

    FROM 'train.parquet'
    WHERE TRY_CAST(srch_co AS DATE)
          <= TRY_CAST(srch_ci AS DATE)
""").fetchdf()

check_out_not_after_check_in

,invalid_rows,invalid_rows_percentage
0,145602,0.3865


что-то очень много строк, посмотрим отедльно на случаи, когда дата заезда и выезда совпадают 

In [24]:
stay_date_quality = con.execute("""
    SELECT
        SUM(
            CASE
                WHEN TRY_CAST(srch_co AS DATE) < TRY_CAST(srch_ci AS DATE)
                THEN 1 ELSE 0
            END
        ) AS check_out_before_check_in,

        ROUND(
            100.0 * SUM(
                CASE
                    WHEN TRY_CAST(srch_co AS DATE) < TRY_CAST(srch_ci AS DATE)
                    THEN 1 ELSE 0
                END
            ) / COUNT(*),
            4
        ) AS check_out_before_check_in_percentage,

        SUM(
            CASE
                WHEN TRY_CAST(srch_co AS DATE) = TRY_CAST(srch_ci AS DATE)
                THEN 1 ELSE 0
            END
        ) AS same_day_check_in_check_out,

        ROUND(
            100.0 * SUM(
                CASE
                    WHEN TRY_CAST(srch_co AS DATE) = TRY_CAST(srch_ci AS DATE)
                    THEN 1 ELSE 0
                END
            ) / COUNT(*),
            4
        ) AS same_day_check_in_check_out_percentage

    FROM 'train.parquet'
""").fetchdf()

stay_date_quality

,check_out_before_check_in,check_out_before_check_in_percentage,same_day_check_in_check_out,same_day_check_in_check_out_percentage
0,798.0,0.0021,144804.0,0.3844


да, так и есть, много записей, когда даты заезда и выезда совпадают. Вообще вроде так нельзя бронить, но допустим, что это окей.

Теперь логично проверить дубли. Но здесь есть нюанс: из-за cnt одинаковые события уже могут быть агрегированы, поэтому сначала просто посчитаем количество полностью одинаковых строк.

In [30]:
duplicate_rows_stats = con.execute("""
    SELECT
        total_rows,
        unique_rows,
        total_rows - unique_rows AS duplicate_rows,
        ROUND(
            100.0 * (total_rows - unique_rows) / total_rows,
            4
        ) AS duplicate_rows_percentage
    FROM (
        SELECT
            (SELECT COUNT(*) FROM 'train.parquet') AS total_rows,
            (
                SELECT COUNT(*)
                FROM (
                    SELECT DISTINCT *
                    FROM 'train.parquet'
                )
            ) AS unique_rows
    )
""").fetchdf()

duplicate_rows_stats

,total_rows,unique_rows,duplicate_rows,duplicate_rows_percentage
0,37670293,37669324,969,0.0026


Теперь проверим признаки, где возможны очевидные ошибки: отрицательные значения, нулевое число взрослых/комнат, странные значения бинарных признаков и т. п.

Сначала посмотрим диапазоны:

In [31]:
value_ranges = con.execute("""
    SELECT
        MIN(is_mobile) AS min_is_mobile,
        MAX(is_mobile) AS max_is_mobile,

        MIN(is_package) AS min_is_package,
        MAX(is_package) AS max_is_package,

        MIN(is_booking) AS min_is_booking,
        MAX(is_booking) AS max_is_booking,

        MIN(srch_adults_cnt) AS min_adults,
        MAX(srch_adults_cnt) AS max_adults,

        MIN(srch_children_cnt) AS min_children,
        MAX(srch_children_cnt) AS max_children,

        MIN(srch_rm_cnt) AS min_rooms,
        MAX(srch_rm_cnt) AS max_rooms,

        MIN(cnt) AS min_cnt,
        MAX(cnt) AS max_cnt,

        MIN(orig_destination_distance) AS min_distance,
        MAX(orig_destination_distance) AS max_distance

    FROM 'train.parquet'
""").fetchdf()

value_ranges

,min_is_mobile,max_is_mobile,min_is_package,max_is_package,min_is_booking,max_is_booking,min_adults,max_adults,min_children,max_children,min_rooms,max_rooms,min_cnt,max_cnt,min_distance,max_distance
0,0,1,0,1,0,1,0,9,0,9,0,8,1,269,0.0056,12407.9022


Бинарные признаки выглядят корректно

cnt тоже выглядит технически корректно.

Ноль детей — абсолютно нормально. А вот 0 взрослых, 0 комнат и особенно 0 взрослых + 0 детей стоит проверить отдельно.

In [39]:
traveler_quality = con.execute("""
    SELECT
        COUNT(*) AS total_rows,

        SUM(
            CASE WHEN srch_adults_cnt = 0 THEN 1 ELSE 0 END
        ) AS zero_adults_rows,

        ROUND(
            100.0 * SUM(
                CASE WHEN srch_adults_cnt = 0 THEN 1 ELSE 0 END
            ) / COUNT(*),
            4
        ) AS zero_adults_percentage,

        SUM(
            CASE WHEN srch_rm_cnt = 0 THEN 1 ELSE 0 END
        ) AS zero_rooms_rows,

        ROUND(
            100.0 * SUM(
                CASE WHEN srch_rm_cnt = 0 THEN 1 ELSE 0 END
            ) / COUNT(*),
            4
        ) AS zero_rooms_percentage,

        SUM(
            CASE
                WHEN srch_adults_cnt = 0
                 AND srch_children_cnt = 0
                THEN 1 ELSE 0
            END
        ) AS zero_travelers_rows,

        ROUND(
            100.0 * SUM(
                CASE
                    WHEN srch_adults_cnt = 0
                     AND srch_children_cnt = 0
                    THEN 1 ELSE 0
                END
            ) / COUNT(*),
            4
        ) AS zero_travelers_percentage

    FROM 'train.parquet'
""").fetchdf()

traveler_quality

,total_rows,zero_adults_rows,zero_adults_percentage,zero_rooms_rows,zero_rooms_percentage,zero_travelers_rows,zero_travelers_percentage
0,37670293,70979.0,0.1884,859.0,0.0023,68634.0,0.1822


И я бы ещё отдельно посмотрела, насколько эти странные случаи вообще доходят до бронирования:

In [38]:
traveler_quality_booking = con.execute("""
    SELECT
        'zero_adults' AS condition,
        COUNT(*) AS rows_count,
        SUM(is_booking) AS booking_rows
    FROM 'train.parquet'
    WHERE srch_adults_cnt = 0

    UNION ALL

    SELECT
        'zero_rooms' AS condition,
        COUNT(*) AS rows_count,
        SUM(is_booking) AS booking_rows
    FROM 'train.parquet'
    WHERE srch_rm_cnt = 0

    UNION ALL

    SELECT
        'zero_travelers' AS condition,
        COUNT(*) AS rows_count,
        SUM(is_booking) AS booking_rows
    FROM 'train.parquet'
    WHERE srch_adults_cnt = 0
      AND srch_children_cnt = 0
""").fetchdf()

traveler_quality_booking

,condition,rows_count,booking_rows
0,zero_adults,70979,5239.0
1,zero_rooms,859,250.0
2,zero_travelers,68634,5076.0


Нулевые бронирования выглядят сомнительно.

Теперь я бы проверила количество уникальных значений в категориальных полях. Это нужно, чтобы понять кардинальность признаков. Например, если hotel_cluster имеет всего около 100 значений, его можно анализировать как обычную категорию, а user_location_city с десятками тысяч значений уже потребует другой подход.

In [44]:
categorical_stats = con.execute("""
    SELECT
        COUNT(DISTINCT site_name) AS site_name_unique,
        COUNT(DISTINCT posa_continent) AS posa_continent_unique,
        COUNT(DISTINCT user_location_country) AS user_location_country_unique,
        COUNT(DISTINCT user_location_region) AS user_location_region_unique,
        COUNT(DISTINCT user_location_city) AS user_location_city_unique,

        COUNT(DISTINCT channel) AS channel_unique,

        COUNT(DISTINCT srch_destination_id) AS destination_unique,
        COUNT(DISTINCT srch_destination_type_id) AS destination_type_unique,

        COUNT(DISTINCT hotel_continent) AS hotel_continent_unique,
        COUNT(DISTINCT hotel_country) AS hotel_country_unique,
        COUNT(DISTINCT hotel_market) AS hotel_market_unique,

        COUNT(DISTINCT hotel_cluster) AS hotel_cluster_unique

    FROM 'train.parquet'
""").fetchdf()

categorical_stats

,site_name_unique,posa_continent_unique,user_location_country_unique,user_location_region_unique,user_location_city_unique,channel_unique,destination_unique,destination_type_unique,hotel_continent_unique,hotel_country_unique,hotel_market_unique,hotel_cluster_unique
0,45,5,237,1008,50447,11,59455,10,7,213,2118,100


Низкая и средняя кардинальность - можно спокойно строить распределения и сравнивать группы:
site_name, posa_continent, channel, srch_destination_type_id, hotel_continent, hotel_cluster.

Высокая кардинальность - бессмысленно строить график по всем значениям, поэтому будем смотреть топ-N, долю топовых категорий и метрики внутри них:
user_location_country, user_location_region, user_location_city, srch_destination_id, hotel_country, hotel_market.
Например, график по всем 59 455 направлениям будет бесполезен, а вот топ-20 направлений по числу взаимодействий + доля booking-строк уже может дать бизнес-смысл.

Нет ли там случайно пользователей, которые по какой-то причине не имеют в логах клика, но при этом есть лог о бронировании отеля? (Нам нужно найти пользователей, у которых есть хотя бы одна строка is_booking = 1, но вообще нет ни одной строки is_booking = 0.)

In [45]:
booking_without_click_users = con.execute("""
    SELECT
        COUNT(*) AS users_with_booking_but_no_click
    FROM (
        SELECT
            user_id
        FROM 'train.parquet'
        GROUP BY user_id
        HAVING
            SUM(CASE WHEN is_booking = 1 THEN 1 ELSE 0 END) > 0
            AND
            SUM(CASE WHEN is_booking = 0 THEN 1 ELSE 0 END) = 0
    )
""").fetchdf()

booking_without_click_users

,users_with_booking_but_no_click
0,0


Ура, таких пользователей нет, значит, все нормально с учетом кликов.